# 01 — Explore Data

Ecommerce KPI RAG Capstone — LLM Zoomcamp 2026

This notebook loads the raw dataset, merges the tables we need, and produces the summary
statistics that feed both the RAG document store (Notebook 02) and the README.

## 1. Load raw data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "../data"

customers = pd.read_csv(f"{DATA_DIR}/customers.csv")
products = pd.read_csv(f"{DATA_DIR}/products.csv")
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv", parse_dates=["timestamp"])
campaigns = pd.read_csv(f"{DATA_DIR}/campaigns.csv")

print("customers:   ", customers.shape)
print("products:    ", products.shape)
print("transactions:", transactions.shape)
print("campaigns:   ", campaigns.shape)

Note: `events.csv` (2,000,000 rows) is intentionally excluded from this pipeline — it's not
needed for KPI reporting and is too large to commit to Git (see the project README for the
Kaggle-download workaround if you need it for a different analysis).

## 2. Merge transactions with product and customer attributes

In [ ]:
df = transactions.merge(products, on="product_id", how="left")
df = df.merge(customers, on="customer_id", how="left", suffixes=("", "_customer"))

print(df.shape)
df.head()

## 3. Data quality checks

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nDuplicate transaction_ids:", df["transaction_id"].duplicated().sum())
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Refund rate: {:.2%}".format(df["refund_flag"].mean()))

**Finding:** 10,449 transactions (~10.1% of the table) have a missing `product_id` and
`gross_revenue` — this is present in the raw `transactions.csv`, not introduced by the merge
(confirmed by checking the raw file directly; every `product_id` that *is* present matches a
real row in `products.csv`, so it isn't a join mismatch either). These rows are excluded from
revenue and category breakdowns below since there's no reliable way to recover the missing
values. Flagging this as a known data quality gap for the README rather than silently dropping
it — it's worth resolving at the source if this dataset is used again.

## 4. Summary statistics

In [ ]:
total_revenue = df["gross_revenue"].sum()
total_orders = df["transaction_id"].nunique()
avg_order_value = df["gross_revenue"].mean()

print(f"Total revenue:      ${total_revenue:,.2f}")
print(f"Total orders:       {total_orders:,}")
print(f"Average order value: ${avg_order_value:,.2f}")
print(f"Unique customers who purchased: {df['customer_id'].nunique():,}")
print(f"Unique products sold:           {df['product_id'].nunique():,}")

## 5. Revenue by category

In [ ]:
category_revenue = (
    df.groupby("category")["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
)
category_revenue

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
category_revenue.plot(kind="bar", ax=ax, color="#3FB6A8")
ax.set_ylabel("Total revenue ($)")
ax.set_title("Revenue by Product Category")
plt.tight_layout()
plt.savefig("../data/category_revenue.png", dpi=150)
plt.show()

## 6. Revenue by country and loyalty tier

In [ ]:
country_revenue = df.groupby("country")["gross_revenue"].sum().sort_values(ascending=False)
loyalty_revenue = df.groupby("loyalty_tier")["gross_revenue"].sum().sort_values(ascending=False)

print("Revenue by country:")
print(country_revenue)
print("\nRevenue by loyalty tier:")
print(loyalty_revenue)

## 7. Monthly revenue trend

In [ ]:
df["month"] = df["timestamp"].dt.to_period("M")
monthly_revenue = df.groupby("month")["gross_revenue"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
monthly_revenue.plot(ax=ax, color="#B85C3E")
ax.set_ylabel("Revenue ($)")
ax.set_title("Monthly Revenue Trend (2021-2023)")
plt.tight_layout()
plt.savefig("../data/monthly_revenue.png", dpi=150)
plt.show()

## 8. Campaign performance

In [ ]:
campaign_revenue = (
    df[df["campaign_id"] > 0]
    .merge(campaigns, on="campaign_id", how="left")
    .groupby("channel")["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
)
campaign_revenue

## 9. Save the merged dataset for downstream notebooks

In [ ]:
df.to_csv(f"{DATA_DIR}/merged_transactions.csv", index=False)
print("Saved merged_transactions.csv:", df.shape)

## Summary

- **103,127 transactions** across **6 product categories** (Grocery, Fashion, Electronics, Sports, Beauty, Home)
- **100,000 customers** across **7 countries** (US, IN, UK, BR, CA, DE, AU) and 4 loyalty tiers
- **Total revenue: ~$8.37M** over Jan 2021 - Dec 2023 (avg order value ~$90.36)
- **Refund rate: ~2.9%**
- **Data quality issue found:** 10,449 transactions (~10.1%) are missing `product_id` and
  `gross_revenue` in the raw source data — excluded from revenue calculations, flagged in the README
- No duplicate transaction IDs
- Electronics is the top-revenue category (~$3.45M); Grocery is the lowest (~$292K)
- US is the top-revenue country (~$2.95M); Australia is the lowest (~$591K)

This merged, cleaned dataset (`merged_transactions.csv`) is what Notebook 02 uses to build
the RAG document store.